In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

# 1.패키지 설치 및 환경
- cohere 회원가입 후 api_key를 추가(COHERE_API_KEY=xxx)
- https://docs.cohere.com/docs/rerank 에서 rerank-multilingual-v3.0 모델을 이용한 rerank

In [ ]:
%pip show scikit-learn

# 2. 환경변수 설정 

In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True) # 커널을 재시작하지 않아도 환경변수 다시 가져오기
OPENAI_LLM_MODEL = "gpt-4o-mini"
UPSTAGE_LLM_MODEL = "solar-pro4"
#OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"
UPSTAGE_EMBEDDING_MODEL = "embedding-query" # 차원수 4096
PINECONE_INDEX_NAME = "better-rag-index"
COHERE_API_KEY = os.getenv("COHERE_API_KEY")


# 3. category 추출하기
- chunk

In [3]:
def get_category():
    return {
        '납세의무': {
            '납세의무': 3, '거주자': 3, '비거주자': 3, '납세의무자': 3, 
            '원천징수': 2, '원천징수의무자': 2, '공동사업자': 2, 
            '상속인': 2, '증여자': 2, '신탁재산': 2
        },

        '세율계산': {
            '세율': 3, '소득세': 3, '과세표준': 3, '산출세액': 3, '세액': 3,
            '결정세액': 2, '세액계산': 2, '기본세율': 2, '세율적용': 2,
            '누진세율': 2, '종합소득세': 2
        },

        '근로소득': {
            '근로소득': 3, '총급여': 3, '급여': 3, '연봉': 3, '임금': 3,
            '근로소득금액': 2, '총급여액': 2, '상여': 2, '수당': 2,
            '봉급': 2, '직장인': 2
        },

        '사업소득': {
            '사업소득': 3, '총수입금액': 3, '필요경비': 3,
            '사업자': 2, '사업소득금액': 2, '결손금': 2, '이월결손금': 2,
            '주택임대소득': 2, '공동사업': 2
        },

        '이자배당': {
            '이자소득': 3, '배당소득': 3, '예금이자': 2, '채권': 2,
            '의제배당': 2, '배당세액공제': 2, '분리과세이자소득': 2,
            '분리과세배당소득': 2
        },

        '양도소득': {
            '양도소득': 3, '자산양도': 2, '부동산양도': 2, '주식양도': 2,
            '양도차익': 2, '취득가액': 2, '양도가액': 2, '양도소득금액': 2
        },

        '연금소득': {
            '연금소득': 3, '연금계좌': 3, '연금저축': 3,
            '퇴직연금': 2, '공적연금': 2, '사적연금': 2, '연금보험': 2,
            '연금수령': 2
        },

        '기타소득': {
            '기타소득': 3, '가상자산': 3, '가상자산소득': 2,
            '상금': 2, '보상금': 2, '종교인소득': 2, '원고료': 2,
            '복권': 1, '당첨금': 1, '발명보상금': 1
        },

        '공제감면': {
            '공제': 3, '소득공제': 3, '세액공제': 3,
            '기본공제': 2, '인적공제': 2, '특별공제': 2, '추가공제': 2,
            '근로소득공제': 2, '연금소득공제': 2, '퇴직소득공제': 2,
            '연금계좌세액공제': 2, '감면': 2
        },

        '비과세': {
            '비과세': 3, '비과세소득': 3, '면제': 2, '세액감면': 2,
            '소득세면제': 2, '복무급여': 1, '실업급여': 1, 
            '출산휴가급여': 1, '장학금': 1
        },

        '신고납부': {
            '신고': 3, '확정신고': 3, '과세표준확정신고': 3, 
            '납부': 3, '중간예납': 2, '가산세': 2,
            '신고기한': 2, '납부기한': 2, '납세지': 2
        },

        '과세기간': {
            '과세기간': 3, '과세연도': 3, '사업연도': 2, 
            '과세기간종료일': 2
        },

        '과세방식': {
            '종합과세': 3, '분리과세': 3, '합산과세': 2, 
            '종합소득과세표준': 2, '분리과세소득': 2, 
            '금융투자소득': 2
        },

        '장부기장': {
            '장부': 3, '복식부기': 3, '간편장부': 2, '기장': 2,
            '장부기록': 2, '증명서류': 2, '기장세액공제': 2
        }
    }

In [4]:
def categorize_content(content, top_k=None):
    category_scores = {}
    """
    content의 카테고리를 분류 - 점수 기반으로 모든 카테고리를 점수순으로 list를 반환(top_k만큼)
        Parameters : 
        -content : 카테고리를 분류할 텍스트
        -top_k : 상위 몇개까지의 카테고리를 반환할지 정수나 None(None이면 모든 카테고리를 반환)
        Returns:
        - 카테고리 리스트(점수순) ex.{'장부기장':30,'과세방식':45}=>['과세방식','장부기장']
    """
    category_keywords = get_category()
    category_score = {}
    # 각 카테고리별 점수계산
    for category, weighted_keywords in category_keywords.items():
#         print(category)
        score = 0
        for keyword,weight in weighted_keywords.items():
            #print(keyword,weight)
            count = content.count(keyword)
            score += count*weight
        if score>0:
            category_scores[category] = score
#     print(category_scores)
    sorted_categories = sorted(category_scores.items(),key=lambda item:item[1],reverse=True)#내림차순
#     print(sorted_categories)
    all_categories = [category[0] for category in sorted_categories]
    # 매칭되는 카테고리가 없을 경우 ["기타"]를 반환
    if not all_categories:
        all_categories = ["기타"]
#     print(all_categories)
    if top_k is not None:
        return all_categories[:top_k]
    else:
        return all_categories
        

In [5]:
content="""거주자 또는 비거주자가 사망하여 그 상속인이 피상속인에 대한 소득세의 납세의무자가 된 경우 그 소득세의 납세지는 그 피상속인ㆍ상속인 또는 납세관리인의 주소지나 거소지 중 상속인 또는 납세관리인이 대통령령으로 정하는 바에 따라 그 관할 세무서장에게 납세지로서 신고하는 장소로 한다.
② 비거주자가 납세관리인을 둔 경우 그 비거주자의 소득세 납세지는 그 국내사업장의 소재지 또는 그 납세관리인의 주소지나 거소지 중 납세관리인이 대통령령으로 정하는 바에 따라 그 관할 세무서장에게 납세지로서 신고하는 장소로 한다.
③ 제1항 또는 제2항에 따른 신고가 있는 때에는 그때부터 그 신고한 장소를 거주자 또는 비거주자의 소득세 납세지로 한다.
④ 제1항이나 제2항에 따른 신고가 없는 경우의 거주자 또는 비거주자의 소득세 납세지는 제6조와 제7조에 따른다.
⑤ 국내에 주소가 없는 공무원 등 대통령령으로 정하는 사람의 소득세 납세지는 대통령령으로 정하는 장소로 한다
 """
print(categorize_content(content))
print(categorize_content("연봉 5천만원인 직장인의 소득세는 얼마에요?"))


['납세의무', '신고납부', '세율계산']
['근로소득', '세율계산']


# 4. langchain 답변생성

In [6]:
from langchain_openai import ChatOpenAI
from langchain_upstage import UpstageEmbeddings
# from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [7]:
# 1. 임베딩 & LLM 초기화
load_dotenv()
llm = ChatOpenAI(model=OPENAI_LLM_MODEL)
embedding = UpstageEmbeddings(model=UPSTAGE_EMBEDDING_MODEL)# pinecone에 저장시 사용했던 임베딩 모델

# 2. vector DB load
# pc = Pinecone()
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=PINECONE_INDEX_NAME
)

In [8]:
# 3. 키워드 사전 활용하여 질문 표준화
keyword_dict  = [
    "사람을 나타내는 표현 -> 거주자",
    "직장인 -> 근로소득이 있는 거주자", 
    "월급쟁이 -> 근로소득이 있는 거주자",
    "회사원 -> 근로소득이 있는 거주자",
    "연봉 -> 종합소득",
    "월급 -> 근로소득",
    "세금 -> 소득세",
    "공제받다 -> 공제를 적용받다",
    "얼마나 내야하나 -> 세액은 얼마인가",
    "계산해줘 -> 계산하면 얼마인가"
]
prompt = ChatPromptTemplate.from_template(f"""사용자의 질문을 보고, 우리의 사전을 참고해서 사용자 질문을
변경해 주세요. 만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지 않아도 됩니다. 오직 질문만 리턴해 주세요.
사전 : {keyword_dict}
질문 : {{question}}""")
# llm.invoke(prompt.invoke({'question':"월급이 높은 남자가 있습니다"}))
keyword_chain = prompt | llm | StrOutputParser()
query = "연봉 5000만원인 회사원의 소득세는 얼마예요?"
normalized_query = keyword_chain.invoke({'question':query})
# keyword_chain.invoke({'question':query})
#keyword_chain.invoke(query)

In [9]:
# 4. Retriver 생성
retriever = vectorStore.as_retriever(
    search_type='similarity',
    search_kwargs = {
        'k':10,
        'filter' : {'category':{'$in':categorize_content(normalized_query)}}
    }
)
documents = retriever.invoke(normalized_query)
# retriever.invoke('종합소득 5000만원인 근로소득이 있는 거주자의 소득세는 세액은 얼마인가요?')

In [10]:
print("순위 재조정 전의 관련문서")
for i, doc in enumerate(documents):
    chunk_id = doc.metadata.get('chunk_id')
    category = doc.metadata.get('category')
    title = doc.metadata.get('title')
    article = doc.metadata.get('article','조항없음')
    print(f'{i+1}.{chunk_id}:{title[:50]}-{category[0]}')

순위 재조정 전의 관련문서
1.chunk_043:소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이 -세율계산
2.chunk_037:소득세 공제 규정 요약 - 일용근로자 공제: 1일 최대 15만원 공제 2018년 개정. --공제감면
3.chunk_048:소득세법 주요 조항 요약: - 근로소득세액공제: 총급여액에 따라 차등 공제 최대 한도 명시-근로소득
4.chunk_049:연금계좌세액공제 - 종합소득자 대상, 연금계좌 납입액의 일정 비율 공제 근로소득자 기준: -연금소득
5.chunk_036:소득공제 및 세액공제 특례 - 결손금 및 이월결손금 공제: 과세기간 결손금 발생 시 먼저 -공제감면
6.chunk_050:소득세 공제 항목: 장애인 의료비, 특정 연령대 및 질환자 의료비, 난임시술 비용, 교육비-근로소득
7.chunk_053:소득세법 공제 및 감면 규정 요약 - 공제 신청 요건: 거주자 신청 시 제1항제4항 적용.-공제감면
8.chunk_058:소득세 결정세액 계산 특례 - 가상자산소득 및 기타소득: 가상자산소득을 제외한 금액에서 2-세율계산
9.chunk_038:소득세법 부양가족 공제 거주자 및 배우자 포함, 직계존속재혼 시 배우자 포함, 직계비속20-공제감면
10.chunk_009:종합소득세 과세표준 계산 - 종합소득금액 계산 시 이자, 배당 등 다양한 소득 합산 후 공-이자배당


## title+page_content기반 rerank함수(cohere)

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
def rerank_by_title(query:str, documents:list, top_k:int=3) -> list:
    '''
    TFid벡터의 cosine 유사도를 사용한 documents 재정렬
    Args:
    - query : 사용자 질문
    - documents : retriever로 검색된 Document 객체 list
    - top_k : 재정렬 후 반환될 문서 갯수
    Returns:
    - title 유사도 기준으로 재정렬된 Document list(0~top_k앞까지)
    '''
    # Document 객체에서 title 추출
    titles = [doc.metadata.get('title') for doc in documents]
    # TF-IDF 벡터화 객체
    vectorizer = TfidfVectorizer(
        analyzer='char', # 한국어는 char단위가 효과적
        ngram_range=(2, 3) # 2~3글자 
    )
    # query와 titles를 함께 벡터화
    all_texts = [query] + titles
    tfidf_matrics = vectorizer.fit_transform(all_texts)
    # print(tfidf_matrics.shape)
    query_vector = tfidf_matrics[0:1] # 0번째 query
    title_vectors = tfidf_matrics[1:]  # 나머지가 titles
    # print(query_vector.shape, title_vectors.shape)
    similarities = cosine_similarity(query_vector, title_vectors)[0] # 1차원
    # 유사도와 문서를 페어링
    score_docs = list(zip(similarities, documents))
    # 유사도 기준으로 내림차순 (score_docs)
    score_docs.sort(key = lambda x : x[0], reverse=True)
    # Documents만 추출
    reranked_docs = [doc for score, doc in score_docs]
    if top_k:
        return reranked_docs[:top_k]
    return reranked_docs

In [12]:
docs = rerank_by_title(normalized_query,documents,3)
print("순위 재조정 전의 관련문서")
for i, doc in enumerate(docs):
    chunk_id = doc.metadata.get('chunk_id')
    category = doc.metadata.get('category')
    title = doc.metadata.get('title')
    article = doc.metadata.get('article','조항없음')
    print(f'{i+1}.{chunk_id}:{title[:50]}-{category[0]}')

순위 재조정 전의 관련문서
1.chunk_043:소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이 -세율계산
2.chunk_037:소득세 공제 규정 요약 - 일용근로자 공제: 1일 최대 15만원 공제 2018년 개정. --공제감면
3.chunk_009:종합소득세 과세표준 계산 - 종합소득금액 계산 시 이자, 배당 등 다양한 소득 합산 후 공-이자배당


In [13]:
# 5. 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 
모른다고 답하세요.
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 6. 검색된 documents를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])
# 7. RAG chain 구성(LCEL 방식)
rag_chain = (
    {
        'context':retriever | format_documents,
        'query' : RunnablePassthrough() # 질문만 그대로 전달 가능
    }
    | prompt
    | llm
    | StrOutputParser()
)
final_chain = keyword_chain | rag_chain

In [14]:
# 8. 실행
query = "연봉 5천만원인 직장인의 소득세는 얼마예요?"
result = final_chain.invoke(query)
print(result)

연봉이 5천만원인 근로소득이 있는 거주자의 소득세는 다음과 같이 계산됩니다.

1. **소득세의 과세표준 구간 확인:**
   - 5천만원은 "1,400만원 초과 5,000만원 이하" 구간에 해당합니다.

2. **소득세 산출 공식 사용:**
   - 이 구간에 대한 세율은 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)입니다.
   - 5,000만원에서 1,400만원을 빼면 3,600만원이 됩니다.

3. **소득세 계산:**
   - (3,600만원 × 15%) = 540만원
   - 전체 소득세 = 84만원 + 540만원 = 624만원

따라서, 연봉 5천만원인 근로소득이 있는 거주자의 소득세는 624만원입니다.


# 5. 참조조항 포함

In [15]:
docs = retriever.invoke(keyword_chain.invoke(query))
def extract_articles_from_docs(documents):
    """
    검색된 문서 documents에서 article 정보를 추출하여 리스트로 반환
    Args:
    - documents : retriever에서 반환된 Document 객체 리스트
    Returns:
        중복 제거된 조항 리스트를 문자로 바꿔서 반환 (ex. '소득세법 제4조, 제60조')
    """
    articles = []
    for doc in documents:
        # metadata에서 article 정보 추출
        article = doc.metadata.get('article', '조항없음')
        article = article.replace('「', '').replace('」', '')
        article_list = article.split(', ')
        articles.extend(article_list)
    # 중복 제거 및 정렬
    unique_article = list(set(articles))
    unique_article.sort(key=lambda x : int(x[:-1]))
    final_article = ['제'+article for article in unique_article if article!='조항없음']
    result = '소득세법 ' + ', '.join(final_article)
    return result
extract_articles_from_docs(docs)

'소득세법 제2조, 제4조, 제6조, 제12조, 제13조, 제14조, 제15조, 제16조, 제17조, 제19조, 제20조, 제21조, 제22조, 제24조, 제26조, 제27조, 제29조, 제31조, 제35조, 제37조, 제39조, 제41조, 제43조, 제45조, 제46조, 제47조, 제48조, 제49조, 제50조, 제51조, 제52조, 제54조, 제55조, 제58조, 제59조, 제62조, 제91조, 제95조, 제122조, 제127조, 제129조, 제133조, 제134조, 제146조, 제156조, 제160조'

In [16]:
def ask_with_reference(query:str, k:int=10, top_k:int=3):
    """
    query의 수정된 query, 답변, 참고조항을 출력하는 함수
    Args:
    - query : 질문
    - k : 검색할 문서의 수(기본값 : 10)
    -top_k : 재배열된 이후 사용될 문서의 수(기본값:3)
    """
    # query 표준화
    normalized_query = keyword_chain.invoke({"question":query})
    # retriever 생성
    retriever = vectorStore.as_retriever(
                    search_type = 'similarity',
                    search_kwargs ={
                        'k':k,
                        'filter':{'category':{'$in':categorize_content(normalized_query)}}
                    }
    )
    # 검색된 문서
    retrieverd_docs = retriever.invoke(normalized_query)
    # RERANK된 문서 추출
    reranked_docs = rerank_by_title(query=normalized_query,
                                   documents=retrieverd_docs,
                                   top_k=top_k)
    # 참고 조항 추출
    referenced_articles = extract_articles_from_docs(reranked_docs)
    # LLM 답변 생성
    template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 
    모른다고 답하세요.
    질문 : {{query}}
    문맥 : {{context}}
    답변 :"""
    prompt = ChatPromptTemplate.from_template(template)
    prompt_chain = prompt | llm | StrOutputParser()
    answer = prompt_chain.invoke({
                'query':normalized_query,
                'context':format_documents(reranked_docs)
            })
    print('='*60)
    print(f'☑️ 질문 : {query}')
    print(f'☑️ 표준화된 질문 : {normalized_query}')
    print('='*60)
    print(f'📝 답변 :{answer}\n')
    print(f'📢 참고 조항 : {referenced_articles}')
    print('**위 답변은 AI에 의해 작성된 답변이므로 약간의 차이가 날 수 있습니다**')

# 7. 사용예시

In [17]:
ask_with_reference("연봉 5천만원인 직장인의 소득세는 얼마에요?")

☑️ 질문 : 연봉 5천만원인 직장인의 소득세는 얼마에요?
☑️ 표준화된 질문 : 연봉 5천만원인 근로소득이 있는 거주자의 소득세는 세액은 얼마인가?
📝 답변 :연봉 5천만원인 근로소득이 있는 거주자의 소득세를 계산해보겠습니다.

1. **과세표준**: 5천만원
2. **적용 세율**:
   - 1,400만원 이하: 과세표준의 6% 
   - 1,400만원 초과 5,000만원 이하: 84만원 + (1,400만원을 초과하는 금액의 15%)

이제 계산해보겠습니다.

- 1,400만원에 대한 세액: 1,400만원 × 6% = 84만원
- 1,400만원 초과분: 5,000만원 - 1,400만원 = 3,600만원
- 3,600만원에 대한 세액: 3,600만원 × 15% = 540만원

따라서, 총 세액은:
84만원 + 540만원 = 624만원

결론적으로, 연봉 5천만원인 근로소득이 있는 거주자의 소득세는 **624만원**입니다.

📢 참고 조항 : 소득세법 제2조, 제6조, 제12조, 제13조, 제14조, 제16조, 제17조, 제19조, 제20조, 제21조, 제24조, 제26조, 제27조, 제29조, 제31조, 제35조, 제37조, 제39조, 제41조, 제43조, 제46조, 제47조, 제50조, 제51조, 제52조, 제54조, 제55조, 제59조, 제95조, 제122조, 제127조, 제129조, 제160조
**위 답변은 AI에 의해 작성된 답변이므로 약간의 차이가 날 수 있습니다**
